In [25]:
import pandas as pd
import numpy as np
import statistics
from pyprojroot import here

In [26]:
INTERIM_DATA_DIR = here() / "data" / "interim"
PROCESSED_DATA_DIR = here() / "data" / "processed"

In [27]:
norm = pd.read_csv(INTERIM_DATA_DIR/"master_norm.csv", encoding="utf-8")

In [28]:
norm.columns.tolist()

['Unnamed: 0',
 'CVEGEO',
 'water_availability',
 'economic_output',
 'n_industy',
 'distance_ciit',
 'dist_to_border_km',
 'dist_min_ports',
 'highways',
 'dist_min_km',
 'gen_min_km',
 'rails',
 'pv_mean_annual_kwh',
 'wind_mean',
 'rainfall',
 'crime']

In [29]:
norm['idx_resources'] = norm[
    ['pv_mean_annual_kwh',
     'wind_mean',
     'water_availability',
     'rainfall']
].mean(axis=1)

norm['idx_energy'] = norm[
    ['dist_min_km',
    'gen_min_km']
].mean(axis=1)

norm['idx_logistics'] = norm[
    ['distance_ciit',
     'dist_to_border_km',
     'dist_min_ports',
     'rails',
     'highways']
].mean(axis=1)

norm['idx_market'] = norm[
    ['n_industy',
     'economic_output']
].mean(axis=1)

norm['idx_security'] = norm[
    ['crime']
].mean(axis=1)

# R.E.L.M.S. index 

In [30]:
subindex_cols_RELMS = [
    'idx_resources',
    'idx_energy',
    'idx_logistics',
    'idx_market',
    'idx_security'
]

weights_RELMS = np.repeat(
    1 / len(subindex_cols_RELMS),
    len(subindex_cols_RELMS)
)

In [31]:
eps = 1e-10

norm['RELMS_index'] = np.exp(
    (
        weights_RELMS *
        np.log(
            norm[subindex_cols_RELMS].values + eps
        )
    ).sum(axis=1)
)

# R.E.L.M. index

In [32]:
subindex_cols_RELM = [
    'idx_resources',
    'idx_energy',
    'idx_logistics',
    'idx_market'
]

weights_RELM = np.repeat(
    1 / len(subindex_cols_RELM),
    len(subindex_cols_RELM)
)

In [33]:
norm['RELM_index'] = np.exp(
    (
        weights_RELM *
        np.log(
            norm[subindex_cols_RELM].values + eps
        )
    ).sum(axis=1)
)

In [34]:
norm['rank'] = (
    norm['RELMS_index']
    .rank(ascending=False, method='dense')
    .astype(int)
)

In [35]:
norm = norm.sort_values(
    by='RELMS_index',
    ascending=False
)

In [36]:
norm.to_csv(PROCESSED_DATA_DIR/
    "RELMS_index.csv",
    index=False,
    encoding="utf-8"
)

print(norm[
    ['CVEGEO',
     'idx_resources',
     'idx_energy',
     'idx_logistics',
     'idx_market',
     'idx_security',
     'RELMS_index',
     'rank',
     'RELM_index'
    ]
].head(20))

      CVEGEO  idx_resources  idx_energy  idx_logistics  idx_market  \
1581   20557       0.743142    0.646419       0.799718    0.481783   
1020   20441       0.761075    0.548407       0.629025    0.579214   
1082   20124       0.722930    0.491018       0.836796    0.611429   
1325   20014       0.669474    0.635193       0.844350    0.534783   
2339   31101       0.781038    0.539279       0.639682    0.681642   
2099   29043       0.690144    0.712129       0.603378    0.543131   
2098   29038       0.676141    0.580943       0.606267    0.570894   
2091   29005       0.694878    0.630627       0.605349    0.727237   
2238   30059       0.643488    0.513268       0.754356    0.583167   
2046   29060       0.510536    0.675736       0.600673    0.499689   
2078   29048       0.512789    0.724628       0.601338    0.585497   
2090   29039       0.653452    0.510264       0.608626    0.616531   
2344   31050       0.779751    0.751339       0.649187    0.864460   
2044   29059       0

In [37]:
df_subindx = norm[
    [
        'CVEGEO',
        'idx_resources',
        'idx_energy',
        'idx_logistics',
        'idx_market',
        'idx_security'
    ]
]

df_subindx.to_csv(PROCESSED_DATA_DIR/
    "RELMS_subindex.csv",
    index=False
)